In [1]:
with open("real_estate_sales_data.txt") as f:
    real_estate_sales = f.read()

In [32]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator=r'\d+\.',
    chunk_size = 100,
    chunk_overlap = 0,
    length_function = len,
    is_separator_regex = True
)

docs = text_splitter.create_documents([real_estate_sales])

In [33]:
docs[0]

Document(metadata={}, page_content='[客户问题] 这个小区交通便利吗？\n[销售回答] 当然了，这个小区距离地铁站只有几分钟的步行距离，而且附近有多条公交线路，非常方便。')

In [34]:
docs[1].page_content

'[客户问题] 我担心楼下太吵。\n[销售回答] 这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。'

In [35]:
len(docs)

70

In [44]:
from langchain.vectorstores import FAISS
from langchain.embeddings.dashscope import DashScopeEmbeddings
import os

embedding = DashScopeEmbeddings(
    model="text-embedding-v2"
)

db = FAISS.from_documents(docs, embedding)

In [45]:
query="小区吵不吵"

In [46]:
answer_list = db.similarity_search(query)

In [48]:
for answer in answer_list:
    print(answer)

page_content='[客户问题] 我担心楼下太吵。
[销售回答] 这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。'
page_content='[客户问题] 我担心楼下的商业活动会很吵。
[销售回答] 我们在规划时就已经考虑到这一点，商业区和居住区有一定的距离和隔音设计。'
page_content='[客户问题] 我喜欢安静，这里噪音大吗？
[销售回答] 我们特意进行了隔音设计，并且小区内部也有绿化带，整体非常安静。'
page_content='[客户问题] 这里会不会很吵？
[销售回答] 我们有良好的隔音设计和规划，内部环境非常宁静。'


In [58]:
topK_retriever = db.as_retriever(search_kwrags={"k": 3})

In [59]:
docs = topK_retriever.invoke(query)
for doc in docs:
    print(doc.page_content + "\n")

[客户问题] 我担心楼下太吵。
[销售回答] 这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。

[客户问题] 我担心楼下的商业活动会很吵。
[销售回答] 我们在规划时就已经考虑到这一点，商业区和居住区有一定的距离和隔音设计。

[客户问题] 我喜欢安静，这里噪音大吗？
[销售回答] 我们特意进行了隔音设计，并且小区内部也有绿化带，整体非常安静。

[客户问题] 这里会不会很吵？
[销售回答] 我们有良好的隔音设计和规划，内部环境非常宁静。



In [60]:
docs = topK_retriever.invoke("你们有没有1000万的豪宅啊？")
for doc in docs:
    print(doc.page_content + "\n")

[客户问题] 这个价位对我来说有点高。
[销售回答] 我们有不同户型和付款方案，一定有适合您预算的。

[客户问题] 都有哪些户型？
[销售回答] 我们有从一室到四室不等的多种户型，定能满足您不同的居住需求。

[客户问题] 有没有健身房？
[销售回答] 当然，我们的小区内有设备齐全的健身房。

[客户问题] 我担心物业费会很高。
[销售回答] 我们的物业费是根据市场和服务水平来设定的，绝对物有所值。



In [76]:
similarity_retriever = db.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"score_threshold": 0.4}
)

In [77]:
docs = similarity_retriever.invoke(query)
for doc in docs:
    print(doc.page_content + "\n")

[客户问题] 我担心楼下太吵。
[销售回答] 这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。



In [78]:
docs = similarity_retriever.invoke(query)

In [79]:
docs[0].page_content

'[客户问题] 我担心楼下太吵。\n[销售回答] 这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。'

In [90]:
from typing import List

def sales(query: str, score_threshold: float=0.4)-> List[str]:
    similarity_retriever = db.as_retriever(
        search_type = "similarity_score_threshold",
        search_kwargs = {"score_threshold": score_threshold}
    )
    docs = similarity_retriever.invoke(query)
    ans_list = [doc.page_content.split("销售回答] ")[-1] for doc in docs]
    return ans_list

In [91]:
query = "我想离医院近点"

print(sales(query))

/opt/anaconda3/envs/ai-spike/lib/python3.10/site-packages/langchain_core/vectorstores/base.py:1082: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='0f47464d-2b03-464e-a785-4ce0af3e0856', metadata={}, page_content='[客户问题] 附近有医院吗？\n[销售回答] 有的，距离我们小区不远就有几家大型综合医院。'), np.float32(0.3283291)), (Document(id='b544d777-693a-495d-9401-cb67a6e05bff', metadata={}, page_content='[客户问题] 附近有医院吗？\n[销售回答] 是的，附近有多家大型医院，医疗资源非常丰富。'), np.float32(0.2830431)), (Document(id='2080e6cc-419c-4700-a3d1-e54dabb23a42', metadata={}, page_content='[客户问题] 我看周围没有学校。\n[销售回答] 其实附近就有几所知名的学校，并且我们也在考虑未来在社区内建立教育设施。'), np.float32(-0.009180546)), (Document(id='9cf2a39c-9466-4887-90fa-f2674d84fbf5', metadata={}, page_content='[客户问题] 附近有地铁站吗？\n[销售回答] 附近就有地铁站，而且有多条公交线路经过，出行非常方便。'), np.float32(-0.021548629))]
  self.vectorstore.similarity_search_with_relevance_scores(
No relevant docs were retrieved using the relevance score threshold 0.4


[]


In [93]:
print(sales(query, 0.3))

['有的，距离我们小区不远就有几家大型综合医院。']


/opt/anaconda3/envs/ai-spike/lib/python3.10/site-packages/langchain_core/vectorstores/base.py:1082: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='0f47464d-2b03-464e-a785-4ce0af3e0856', metadata={}, page_content='[客户问题] 附近有医院吗？\n[销售回答] 有的，距离我们小区不远就有几家大型综合医院。'), np.float32(0.3283291)), (Document(id='b544d777-693a-495d-9401-cb67a6e05bff', metadata={}, page_content='[客户问题] 附近有医院吗？\n[销售回答] 是的，附近有多家大型医院，医疗资源非常丰富。'), np.float32(0.2830431)), (Document(id='2080e6cc-419c-4700-a3d1-e54dabb23a42', metadata={}, page_content='[客户问题] 我看周围没有学校。\n[销售回答] 其实附近就有几所知名的学校，并且我们也在考虑未来在社区内建立教育设施。'), np.float32(-0.009180546)), (Document(id='9cf2a39c-9466-4887-90fa-f2674d84fbf5', metadata={}, page_content='[客户问题] 附近有地铁站吗？\n[销售回答] 附近就有地铁站，而且有多条公交线路经过，出行非常方便。'), np.float32(-0.021548629))]
  self.vectorstore.similarity_search_with_relevance_scores(


In [95]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(model="qwen-plus",
        api_key = os.getenv("DASHSCOPE_API_KEY"),
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1",
        temperature=0.2, 
        max_tokens=5000,
        verbose=True
)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = db.as_retriever(
        search_type = "similarity_score_threshold",
        search_kwargs = {"score_threshold": 0.4}
    )
)

In [97]:
qa_chain.invoke({"query": "你们小区有200万的房子吗？"})

No relevant docs were retrieved using the relevance score threshold 0.4


{'query': '你们小区有200万的房子吗？',
 'result': '我不了解具体小区的房价情况，因为我是阿里巴巴集团旗下的语言模型，没有实时的房产数据。如果你想知道某个小区是否有200万的房子，建议你查看房产中介网站或者咨询当地的房产中介。'}

In [98]:
qa_chain.invoke({"query": "小区吵不吵"})

{'query': '小区吵不吵',
 'result': '这个小区特别注重居住体验，我们有良好的隔音设计，并且小区内部规划了绿化区域，可以有效降低噪音。所以相对来说，不会很吵。'}